In this ection we will be working with a variation of the [BBC News Classification Dataset](https://www.kaggle.com/c/learn-ai-bbc/overview) which contains 2225 examples of news articles with their respective categories.

Lets get started

In [6]:

import csv
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [13]:
import os


with open('/bbc-text.csv','r') as csvfile:
  print(f"First line(header) looks like this :\n\n{csvfile.readline()}")
  print(f"Each data point looks like this:\n\n{csvfile.readline()}")

First line(header) looks like this :

category,text

Each data point looks like this:

tech,tv future in the hands of viewers with home theatre systems  plasma high-definition tvs  and digital video recorders moving into the living room  the way people watch tv will be radically different in five years  time.  that is according to an expert panel which gathered at the annual consumer electronics show in las vegas to discuss how these new technologies will impact one of our favourite pastimes. with the us leading the trend  programmes and other content will be delivered to viewers via home networks  through cable  satellite  telecoms companies  and broadband service providers to front rooms and portable devices.  one of the most talked-about technologies of ces has been digital and personal video recorders (dvr and pvr). these set-top boxes  like the us s tivo and the uk s sky+ system  allow people to record  store  play  pause and forward wind tv programmes when they want.  essentially

As you can see, each data point is composed of the category of the news article followed by a comma and then the actual text of the article.

# Removing Stopwords

One important step when working with text data is to remove the **stopwords** from it. These are the most common words in the language and they rarely provide useful information for the classification the process.

This function should receive a string and return another string that excludes all of the stopwords provided.

In [14]:
def remove_stopwords(sentence):
    stopwords = ["a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "as", "at", "be", "because", "been", "before", "being", "below", "between", "both", "but", "by", "could", "did", "do", "does", "doing", "down", "during", "each", "few", "for", "from", "further", "had", "has", "have", "having", "he", "he'd", "he'll", "he's", "her", "here", "here's", "hers", "herself", "him", "himself", "his", "how", "how's", "i", "i'd", "i'll", "i'm", "i've", "if", "in", "into", "is", "it", "it's", "its", "itself", "let's", "me", "more", "most", "my", "myself", "nor", "of", "on", "once", "only", "or", "other", "ought", "our", "ours", "ourselves", "out", "over", "own", "same", "she", "she'd", "she'll", "she's", "should", "so", "some", "such", "than", "that", "that's", "the", "their", "theirs", "them", "themselves", "then", "there", "there's", "these", "they", "they'd", "they'll", "they're", "they've", "this", "those", "through", "to", "too", "under", "until", "up", "very", "was", "we", "we'd", "we'll", "we're", "we've", "were", "what", "what's", "when", "when's", "where", "where's", "which", "while", "who", "who's", "whom", "why", "why's", "with", "would", "you", "you'd", "you'll", "you're", "you've", "your", "yours", "yourself", "yourselves" ]
    sentence = sentence.lower()

    words = sentence.split()
    results_words = [word for word in words if word not in stopwords]
    sentence = ' '.join(results_words)

    return sentence

In [16]:
remove_stopwords("I am about to go to the store and get any snack")

'go store get snack'

# Reading the Raw data:

Now you need to read the data from the csv file. To do so, complete the `parse_data_from_file` function.

A couple of things to note:

- ou should omit the first line as it contains the headers and not data points.
- There is no need to save the data points as numpy arrays, regular lists is fine.
- To read from csv files use `csv.reader` by passing the appropriate arguments.
- `csv.reader` returns an iterable that returns each row in every iteration. So the label can be accessed via row[0] and the text via row[1].
- Use the `remove_stopwords` function in each sentence.

In [22]:
def parse_data_from_file(filename):
  sentences = []
  labels = []
  with open(filename, 'r') as csvfile:
    reader = csv.reader(csvfile, delimiter=',')
    next(reader,None)

    for row in reader:
      labels.append(remove_stopwords(row[0])) # Corrected function call
      sentences.append(remove_stopwords(row[1]))

    return sentences, labels

In [23]:
sentences, labels = parse_data_from_file('/root/.cache/kagglehub/competitions/learn-ai-bbc/BBC News Train.csv') # Corrected function name and file path

print(f"There are {len(sentences)} sentences in the dataset.\n")
print(f"First sentence has {len(sentences[0].split())} words (after removing stopwords).\n")
print(f"There are {len(labels)} labels in the dataset.\n")
print(f"The first 5 labels are {labels[:5]}")

There are 1490 sentences in the dataset.

First sentence has 203 words (after removing stopwords).

There are 1490 labels in the dataset.

The first 5 labels are ['1833', '154', '1101', '1976', '917']


# Using Tokenizer

Now it is time to tokenize the sentences of the dataset.

Complete the `fit_tokenizer` below.

This function should receive the list of sentences as input and return a Tokenizer that has been fitted to those sentences. You should also define the "Out of Vocabulary" token as `<OOV>`.

In [36]:
def fit_tokenizer(sentences):
  tokenizer = Tokenizer(oov_token = '<OOV>')
  tokenizer.fit_on_texts(sentences)
  return tokenizer

In [37]:
tokenizer = fit_tokenizer(sentences)
word_index = tokenizer.word_index

print(f"Vocabulary contains {len(word_index)} words\n")
print("<OOV> token included in vocabulary" if "<OOV>" in word_index else "<OOV> token NOT included in vocabulary")

Vocabulary contains 24963 words

<OOV> token included in vocabulary


In [38]:
def get_padded_sequences(tokenizer, sentences):
    sequences = tokenizer.texts_to_sequences(sentences)

    # Pad the sequences using the post padding strategy
    padded_sequences = pad_sequences(sequences, padding='post')
    ### END CODE HERE

    return padded_sequences

In [39]:
padded_sequences = get_padded_sequences(tokenizer, sentences)
print(f"First padded sequence looks like this: \n\n{padded_sequences[0]}\n")
print(f"Numpy array of all sequences has shape: {padded_sequences.shape}\n")
print(f"This means there are {padded_sequences.shape[0]} sequences in total and each one has a size of {padded_sequences.shape[1]}")

First padded sequence looks like this: 

[1322 1180  592 ...    0    0    0]

Numpy array of all sequences has shape: (1490, 1881)

This means there are 1490 sequences in total and each one has a size of 1881


In [32]:
def tokenize_labels(labels):
  label_tokenizer = Tokenizer()
  label_tokenizer.fit_on_texts(labels)
  label_word_index = label_tokenizer.word_index
  label_sequences = label_tokenizer.texts_to_sequences(labels)
  return label_sequences, label_word_index

In [33]:
label_sequences, label_word_index = tokenize_labels(labels)
print(f"Vocabulary of labels looks like this {label_word_index}\n")
print(f"First ten sequences {label_sequences[:10]}\n")

Vocabulary of labels looks like this {'1833': 1, '154': 2, '1101': 3, '1976': 4, '917': 5, '1582': 6, '651': 7, '1797': 8, '2034': 9, '1866': 10, '1683': 11, '1153': 12, '1028': 13, '812': 14, '707': 15, '1588': 16, '342': 17, '486': 18, '1344': 19, '1552': 20, '1547': 21, '177': 22, '1785': 23, '1617': 24, '405': 25, '1561': 26, '702': 27, '1026': 28, '1527': 29, '1503': 30, '1951': 31, '1407': 32, '2002': 33, '2100': 34, '466': 35, '687': 36, '1009': 37, '805': 38, '771': 39, '1532': 40, '2205': 41, '2000': 42, '953': 43, '1394': 44, '1522': 45, '455': 46, '593': 47, '590': 48, '277': 49, '90': 50, '904': 51, '527': 52, '1763': 53, '42': 54, '1364': 55, '1418': 56, '643': 57, '40': 58, '1518': 59, '2046': 60, '464': 61, '180': 62, '476': 63, '2017': 64, '315': 65, '96': 66, '1079': 67, '947': 68, '1742': 69, '972': 70, '210': 71, '2144': 72, '1766': 73, '1971': 74, '1303': 75, '1638': 76, '79': 77, '1055': 78, '1804': 79, '1929': 80, '371': 81, '445': 82, '105': 83, '1297': 84, '1932